### Домашнее задание 3 - 30 баллов

В этом задании вам предстоит дообучить трансформерную модель для NER-задачи в различных форматах:

1. Обучите NER-модель

- Загрузите набор данных [gusevski/factrueval2016](https://huggingface.co/datasets/gusevski/factrueval2016) или [factRuEval-2016](https://github.com/dialogue-evaluation/factRuEval-2016) - **2 балла**
- Подготовьте набор данных для обучения (токенизация, выравнинивание меток и т.п.) - - **3 балла**
- Дообучите подходящую на ваш взгляд энкодерную модель на train-части корпуса для решения NER-задачи, сделайте замеры качества NER-метрик до и после дообучения - **3 балла**

2. Попробуйте улучшить качество модели следующими способами (из оригинального чекпойнта модели с HF):
- Предварительно дообучите на train-части в MLM режиме, а потом дообучите на NER-задачу - **5 баллов**
- Для улучшения потенциального качества MLM попробуйте не стандартное случайное маскирование отдельных токенов, а использование DataCollatorForWholeWordMask - маскирование целых слов, или Concept masking - маскирование отдельных **NER-сущностей** -  **5 баллов**
- Сгенерируйте синтетическую разметку* подходящего**, на ваш взгляд, корпуса большой и умной моделью для русскоязычного NER***, а затем использовав ее для дообучения вашего энкодера вместе с основным набором данных - **5 баллов**

3. Финально сравните результаты различных подходов, сделайте выводы, опишите, что можно улучшить в дальнейших экспериментах - **3 балла**

*прогоните датасет через NER-модель, получите ее предсказания и используйте их в качестве разметки

**Можно использовать уже знакомый вам датасет lenta-ru, или более релеватные по домену, объем данных лучше взять от 10_000 текстов

***Например, можно взять модель модель DeepPavlov ner_collection3_bert. Инструкция по запуску есть в [документации](https://docs.deeppavlov.ai/en/master/features/models/NER.html), или другую понравившуюся вам модель. Главное - чтобы модель знала русский и умела выделять нужные сущности с качеством лучше вашего энкодера
 

**Общее**

- Принимаемые решения обоснованы (почему выбрана определенная архитектура/гиперпараметр/оптимизатор/преобразование и т.п.), по ходу работы присутствуют комментарии и выводы - **3 балла**
- Обеспечена воспроизводимость решения: зафиксированы random_state, ноутбук воспроизводится от начала до конца без ошибок - **1 балл**

In [1]:
! pip -q install -U "transformers==4.46.3" "datasets==3.1.0" "evaluate==0.4.3" "seqeval==1.2.2" "accelerate==1.0.1" "sentencepiece==0.2.0"


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python3.12 -m pip install --upgrade pip


In [39]:
import os
import gc
import random
from dataclasses import dataclass

import numpy as np
import pandas as pd
import torch

from datasets import load_dataset, Dataset, DatasetDict, concatenate_datasets
import evaluate
from pathlib import Path
from tqdm.auto import tqdm
from tqdm.auto import trange

from transformers import (
    AutoTokenizer,
    AutoConfig,
    AutoModelForTokenClassification,
    AutoModelForMaskedLM,
    DataCollatorForTokenClassification,
    DataCollatorForLanguageModeling,
    DataCollatorForWholeWordMask,
    TrainingArguments,
    Trainer,
    pipeline,
    set_seed,
)

In [3]:
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

BASE_MODEL = "DeepPavlov/rubert-base-cased"

MAX_LENGTH = 256
BATCH_SIZE = 16
MLM_BATCH_SIZE = 16
NER_EPOCHS = 5
MLM_EPOCHS = 2

OUTPUT_ROOT = "./hw3_ner_runs"
os.makedirs(OUTPUT_ROOT, exist_ok=True)

print("device:", DEVICE)
print("MAX_LENGTH:", MAX_LENGTH)

device: cuda
MAX_LENGTH: 256


## Загрузка данных и токенизация

In [4]:
ds_raw = load_dataset("gusevski/factrueval2016")
ds_raw

README.md: 0.00B [00:00, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


train_data.json:   0%|          | 0.00/7.62M [00:00<?, ?B/s]

dev_data.json:   0%|          | 0.00/2.48M [00:00<?, ?B/s]

test_data.json:   0%|          | 0.00/2.57M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['data'],
        num_rows: 1
    })
    validation: Dataset({
        features: ['data'],
        num_rows: 1
    })
    test: Dataset({
        features: ['data'],
        num_rows: 1
    })
})

In [5]:
def unwrap_split(split_ds):
    rows = split_ds[0]["data"]
    return Dataset.from_list(rows)

ds = DatasetDict({
    "train": unwrap_split(ds_raw["train"]),
    "validation": unwrap_split(ds_raw["validation"]),
    "test": unwrap_split(ds_raw["test"]),
})

ds

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'length', 'ner_tags_str', 'ner_tags'],
        num_rows: 7746
    })
    validation: Dataset({
        features: ['id', 'tokens', 'length', 'ner_tags_str', 'ner_tags'],
        num_rows: 2582
    })
    test: Dataset({
        features: ['id', 'tokens', 'length', 'ner_tags_str', 'ner_tags'],
        num_rows: 2582
    })
})

In [6]:
print(ds["train"][0])

# получаем список меток

all_labels = set()
for split in ds:
    for row in ds[split]["ner_tags_str"]:
        all_labels.update(row)

label_names = sorted(all_labels)

if "O" in label_names:
    label_names = ["O"] + [x for x in label_names if x != "O"]

id2label = {i: label for i, label in enumerate(label_names)}
label2id = {label: i for i, label in id2label.items()}

print("num labels:", len(label_names))
print(label_names)

{'id': 0, 'tokens': ['"', 'Если', 'Миронов', 'занял', 'столь', 'оппозиционную', 'позицию', ',', 'то', 'мне', 'представляется', ',', 'что', 'для', 'него', 'было', 'бы', 'порядочным', 'и', 'правильным', 'уйти', 'в', 'отставку', 'с', 'занимаемого', 'им', 'поста', ',', 'поста', ',', 'который', 'предоставлен', 'ему', 'сегодня', '"', 'Единой', 'Россией', "''", 'и', 'никем', 'больше', "''", ',', '-', 'заключает', 'Исаев', '.'], 'length': 47, 'ner_tags_str': ['O', 'O', 'B-PER', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-ORG', 'I-ORG', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-PER', 'O'], 'ner_tags': [0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 4, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0]}
num labels: 7
['O', 'B-LOC', 'B-ORG', 'B-PER', 'I-LOC', 'I-ORG', 'I-PER']


In [7]:
# проверяем что числовые совпадают со строковыми
sample = ds["train"][0]

pairs = list(zip(sample["tokens"], sample["ner_tags"], sample["ner_tags_str"]))
for x in pairs[:20]:
    print(x)

('"', 0, 'O')
('Если', 0, 'O')
('Миронов', 1, 'B-PER')
('занял', 0, 'O')
('столь', 0, 'O')
('оппозиционную', 0, 'O')
('позицию', 0, 'O')
(',', 0, 'O')
('то', 0, 'O')
('мне', 0, 'O')
('представляется', 0, 'O')
(',', 0, 'O')
('что', 0, 'O')
('для', 0, 'O')
('него', 0, 'O')
('было', 0, 'O')
('бы', 0, 'O')
('порядочным', 0, 'O')
('и', 0, 'O')
('правильным', 0, 'O')


In [8]:
# строим отображение id-label напрямую из датасета

tmp_map = {}

for split in ds:
    for nums, strs in zip(ds[split]["ner_tags"], ds[split]["ner_tags_str"]):
        for n, s in zip(nums, strs):
            if n in tmp_map and tmp_map[n] != s:
                raise ValueError(f"Inconsistent mapping for tag id {n}: {tmp_map[n]} vs {s}")
            tmp_map[n] = s

id2label = dict(sorted(tmp_map.items()))
label2id = {v: k for k, v in id2label.items()}

print("id2label =", id2label)
print("label2id =", label2id)

id2label = {0: 'O', 1: 'B-PER', 2: 'I-PER', 3: 'B-ORG', 4: 'I-ORG', 5: 'B-LOC', 6: 'I-LOC'}
label2id = {'O': 0, 'B-PER': 1, 'I-PER': 2, 'B-ORG': 3, 'I-ORG': 4, 'B-LOC': 5, 'I-LOC': 6}


In [9]:
# токенизатор
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer

tokenizer_config.json:   0%|          | 0.00/24.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/642 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

BertTokenizerFast(name_or_path='DeepPavlov/rubert-base-cased', vocab_size=119547, model_max_length=1000000000000000019884624838656, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, clean_up_tokenization_spaces=True),  added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}

In [10]:
# выравнивание меток после сабтокенизации
def tokenize_and_align_labels(examples):
    tokenized = tokenizer(
        examples["tokens"],
        is_split_into_words=True,
        truncation=True,
        max_length=MAX_LENGTH,
    )

    aligned_labels = []

    for i in range(len(examples["tokens"])):
        word_ids = tokenized.word_ids(batch_index=i)
        word_labels = examples["ner_tags"][i]

        prev_word_id = None
        label_ids = []

        for word_id in word_ids:
            if word_id is None:
                label_ids.append(-100)
            elif word_id != prev_word_id:
                label_ids.append(word_labels[word_id])
            else:
                cur_label = id2label[word_labels[word_id]]
                if cur_label.startswith("B-"):
                    inside_label = "I-" + cur_label[2:]
                    label_ids.append(label2id.get(inside_label, word_labels[word_id]))
                else:
                    label_ids.append(word_labels[word_id])

            prev_word_id = word_id

        aligned_labels.append(label_ids)

    tokenized["labels"] = aligned_labels
    return tokenized

# токенизация
tokenized_ds = ds.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=ds["train"].column_names,
)

tokenized_ds

Map:   0%|          | 0/7746 [00:00<?, ? examples/s]

Map:   0%|          | 0/2582 [00:00<?, ? examples/s]

Map:   0%|          | 0/2582 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 7746
    })
    validation: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 2582
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 2582
    })
})

In [11]:
sample = tokenized_ds["train"][0]

print(sample["input_ids"][:20])
print(sample["labels"][:20])

decoded_tokens = tokenizer.convert_ids_to_tokens(sample["input_ids"][:40])
decoded_labels = [id2label[x] if x != -100 else "IGN" for x in sample["labels"][:40]]

for tok, lab in zip(decoded_tokens, decoded_labels):
    print(f"{tok:20s} {lab}")

[101, 108, 10830, 37027, 11532, 18488, 119392, 18099, 128, 3815, 16740, 34246, 128, 1997, 2748, 7268, 3332, 1655, 10040, 20422]
[-100, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[CLS]                IGN
"                    O
Если                 O
Миронов              B-PER
занял                O
столь                O
оппозиционную        O
позицию              O
,                    O
то                   O
мне                  O
представляется       O
,                    O
что                  O
для                  O
него                 O
было                 O
бы                   O
поряд                O
##очным              O
и                    O
правильным           O
уйти                 O
в                    O
отставку             O
с                    O
заним                O
##аемого             O
им                   O
поста                O
,                    O
поста                O
,                    O
который              O
предоставлен        

- спецтокен [CLS] получает -100
- сущности сохраняются корректно
- сабтокены вроде ##очным получают ту же метку, что и исходное слово

## baseline для NER

In [12]:
# метрики для NER
seqeval = evaluate.load("seqeval")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    true_preds = []
    true_labels = []

    for pred_row, label_row in zip(preds, labels):
        cur_preds = []
        cur_labels = []

        for p, l in zip(pred_row, label_row):
            if l != -100:
                cur_preds.append(id2label[p])
                cur_labels.append(id2label[l])

        true_preds.append(cur_preds)
        true_labels.append(cur_labels)

    metrics = seqeval.compute(predictions=true_preds, references=true_labels)

    return {
        "precision": metrics["overall_precision"],
        "recall": metrics["overall_recall"],
        "f1": metrics["overall_f1"],
        "accuracy": metrics["overall_accuracy"],
    }

In [13]:
# функция для сборки baseline
def build_ner_model(model_name_or_path):
    config = AutoConfig.from_pretrained(
        model_name_or_path,
        num_labels=len(id2label),
        id2label=id2label,
        label2id=label2id,
    )

    model = AutoModelForTokenClassification.from_pretrained(
        model_name_or_path,
        config=config,
        ignore_mismatched_sizes=True,
    )
    return model

In [14]:
# функция обучения baseline
def train_ner_model(
    model_name_or_path,
    train_dataset,
    eval_dataset,
    run_name,
    learning_rate=2e-5,
    num_train_epochs=5,
    batch_size=16,
):
    out_dir = os.path.join(OUTPUT_ROOT, run_name)

    model = build_ner_model(model_name_or_path)

    args = TrainingArguments(
        output_dir=out_dir,
        evaluation_strategy="epoch",
        save_strategy="epoch",
        logging_strategy="steps",
        logging_steps=100,
        learning_rate=learning_rate,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        num_train_epochs=num_train_epochs,
        weight_decay=0.01,
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,
        report_to="none",
        fp16=torch.cuda.is_available())

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        tokenizer=tokenizer,
        data_collator=DataCollatorForTokenClassification(tokenizer),
        compute_metrics=compute_metrics)

    metrics_before = trainer.evaluate()
    trainer.train()
    metrics_after = trainer.evaluate()

    trainer.save_model(out_dir)
    tokenizer.save_pretrained(out_dir)

    del model, trainer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return {
        "run_name": run_name,
        "before_f1": metrics_before["eval_f1"],
        "after_f1": metrics_after["eval_f1"],
        "before_precision": metrics_before["eval_precision"],
        "after_precision": metrics_after["eval_precision"],
        "before_recall": metrics_before["eval_recall"],
        "after_recall": metrics_after["eval_recall"],
        "before_accuracy": metrics_before["eval_accuracy"],
        "after_accuracy": metrics_after["eval_accuracy"],
        "model_path": out_dir}

In [15]:
baseline_result = train_ner_model(
    model_name_or_path=BASE_MODEL, #базовый чекпойнт rubert
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["validation"],
    run_name="baseline_ner",
    learning_rate=2e-5, #стандартный небольшой lr для finetuning трансформера
    num_train_epochs=NER_EPOCHS, #5 эпох
    batch_size=BATCH_SIZE, #размер батча=16
)

baseline_result

pytorch_model.bin:   0%|          | 0.00/714M [00:00<?, ?B/s]

Some weights of BertForTokenClassification were not initialized from the model checkpoint at DeepPavlov/rubert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/home/mlcore/conda/lib/python3.12/site-packages/transformers/training_args.py:1568: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
/tmp/ipykernel_363/3265979145.py:33: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Model Preparation Time,Precision,Recall,F1,Accuracy
1,0.032600,0.025884,0.003000,0.965471,0.971028,0.968242,0.992749
2,0.015300,0.020056,0.003000,0.968549,0.980814,0.974643,0.994807
3,0.007000,0.021452,0.003000,0.979931,0.983691,0.981808,0.995785
4,0.003800,0.022483,0.003000,0.981641,0.984843,0.983239,0.996068
5,0.003300,0.021847,0.003000,0.981801,0.983308,0.982554,0.995935


Error during conversion: ChunkedEncodingError(ProtocolError("Connection broken: InvalidChunkLength(got length b'', 0 bytes read)", InvalidChunkLength(got length b'', 0 bytes read)))


model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

{'run_name': 'baseline_ner',
 'before_f1': 0.0186680873734683,
 'after_f1': 0.9832391533378029,
 'before_precision': 0.010500323640112195,
 'after_precision': 0.9816408491107286,
 'before_recall': 0.08403683806600154,
 'after_recall': 0.9848426707597852,
 'before_accuracy': 0.16342005707838322,
 'after_accuracy': 0.9960675648768833,
 'model_path': './hw3_ner_runs/baseline_ner'}

## MLM-предобучение с обычным random masking

In [16]:
# собираем тексты для млм
def join_tokens(example):
    return {"text_for_mlm": " ".join(example["tokens"])}

mlm_raw = ds.map(join_tokens)
mlm_raw

Map:   0%|          | 0/7746 [00:00<?, ? examples/s]

Map:   0%|          | 0/2582 [00:00<?, ? examples/s]

Map:   0%|          | 0/2582 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'length', 'ner_tags_str', 'ner_tags', 'text_for_mlm'],
        num_rows: 7746
    })
    validation: Dataset({
        features: ['id', 'tokens', 'length', 'ner_tags_str', 'ner_tags', 'text_for_mlm'],
        num_rows: 2582
    })
    test: Dataset({
        features: ['id', 'tokens', 'length', 'ner_tags_str', 'ner_tags', 'text_for_mlm'],
        num_rows: 2582
    })
})

In [17]:
# токенизируем
def tokenize_for_mlm(examples):
    return tokenizer(
        examples["text_for_mlm"],
        truncation=True,
        max_length=MAX_LENGTH,
        return_special_tokens_mask=True,
    )

mlm_tokenized = mlm_raw.map(
    tokenize_for_mlm,
    batched=True,
    remove_columns=mlm_raw["train"].column_names,
)

mlm_tokenized

Map:   0%|          | 0/7746 [00:00<?, ? examples/s]

Map:   0%|          | 0/2582 [00:00<?, ? examples/s]

Map:   0%|          | 0/2582 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'special_tokens_mask'],
        num_rows: 7746
    })
    validation: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'special_tokens_mask'],
        num_rows: 2582
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'special_tokens_mask'],
        num_rows: 2582
    })
})

In [18]:
# функция млм обучения
def pretrain_mlm(
    model_name_or_path,
    train_dataset,
    eval_dataset,
    run_name,
    collator,
    learning_rate=5e-5,
    num_train_epochs=2,
    batch_size=16,
):
    out_dir = os.path.join(OUTPUT_ROOT, run_name)

    model = AutoModelForMaskedLM.from_pretrained(model_name_or_path)

    args = TrainingArguments(
        output_dir=out_dir,
        evaluation_strategy="epoch",
        save_strategy="epoch",
        logging_strategy="steps",
        logging_steps=100,
        learning_rate=learning_rate,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        num_train_epochs=num_train_epochs,
        weight_decay=0.01,
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        report_to="none",
        fp16=torch.cuda.is_available(),
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        tokenizer=tokenizer,
        data_collator=collator,
    )

    metrics_before = trainer.evaluate()
    trainer.train()
    metrics_after = trainer.evaluate()

    trainer.save_model(out_dir)
    tokenizer.save_pretrained(out_dir)

    del model, trainer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return {
        "run_name": run_name,
        "mlm_loss_before": metrics_before["eval_loss"],
        "mlm_loss_after": metrics_after["eval_loss"],
        "model_path": out_dir,
    }

In [19]:
random_mask_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=0.15,
)

mlm_random_result = pretrain_mlm(
    model_name_or_path=BASE_MODEL, # исходный rubert
    train_dataset=mlm_tokenized["train"],
    eval_dataset=mlm_tokenized["validation"],
    run_name="mlm_random_masking",
    collator=random_mask_collator, # случайное маскирование токенов с вероятностью 0.15
    learning_rate=5e-5, #для млм можно взять lr чуть выше
    num_train_epochs=MLM_EPOCHS, # 2 эпохи, так как это промежуточная доменная адаптация энкодера
    batch_size=MLM_BATCH_SIZE, # 16
)

mlm_random_result

BertForMaskedLM has generative capabilities, as `prepare_inputs_for_generation` is explicitly overwritten. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other related functions.
  - If you're using `trust_remote_code=True`, you can get rid of this warning by loading the model with an auto class. See https://huggingface.co/docs/transformers/en/model_doc/auto#auto-classes
  - If you are the owner of the model architecture code, please modify your model class such that it inherits from `GenerationMixin` (after `PreTrainedModel`, otherwise you'll get an exception).
  - If you are not the owner of the model architecture class, please contact the model code owner to update it.
/home/mlcore/conda/lib/python3.12/site-packages/transformers/training_args.py:1568: FutureWarning: `evaluation_strategy` is deprecated and will be removed in versio

Epoch,Training Loss,Validation Loss,Model Preparation Time
1,1.736700,1.767285,0.002800
2,1.563100,1.695761,0.002800


There were missing keys in the checkpoint model loaded: ['cls.predictions.decoder.weight', 'cls.predictions.decoder.bias'].


{'run_name': 'mlm_random_masking',
 'mlm_loss_before': 1.7442635297775269,
 'mlm_loss_after': 1.675248384475708,
 'model_path': './hw3_ner_runs/mlm_random_masking'}

## Дообучение NER после MLM

In [21]:
mlm_then_ner_result = train_ner_model(
    model_name_or_path=mlm_random_result["model_path"],  #инициализация из чекпойнта
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["validation"], 
    run_name="mlm_random_then_ner",
    learning_rate=2e-5, # стандартный lr для finetuning NER
    num_train_epochs=NER_EPOCHS, 
    batch_size=BATCH_SIZE,
)

mlm_then_ner_result

Some weights of BertForTokenClassification were not initialized from the model checkpoint at ./hw3_ner_runs/mlm_random_masking and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/home/mlcore/conda/lib/python3.12/site-packages/transformers/training_args.py:1568: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
/tmp/ipykernel_363/3265979145.py:33: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Model Preparation Time,Precision,Recall,F1,Accuracy
1,0.032300,0.027744,0.004900,0.958546,0.971604,0.965031,0.992301
2,0.016300,0.018648,0.004900,0.973478,0.978895,0.976179,0.995454
3,0.006800,0.021376,0.004900,0.974120,0.982157,0.978122,0.995503
4,0.005200,0.023231,0.004900,0.974657,0.981389,0.978011,0.995437
5,0.003800,0.023199,0.004900,0.976895,0.981581,0.979232,0.995520


{'run_name': 'mlm_random_then_ner',
 'before_f1': 0.018799569176539705,
 'after_f1': 0.9792324624365968,
 'before_precision': 0.010468235448062286,
 'after_precision': 0.9768951689898797,
 'before_recall': 0.0920951650038373,
 'after_recall': 0.9815809669992326,
 'before_accuracy': 0.12995287714873566,
 'after_accuracy': 0.9955200106192341,
 'model_path': './hw3_ner_runs/mlm_random_then_ner'}

In [22]:
# сравниваем baseline и mlm + ner
compare_df = pd.DataFrame([
    {
        "experiment": "baseline NER",
        "f1_before": baseline_result["before_f1"],
        "f1_after": baseline_result["after_f1"],
        "precision_after": baseline_result["after_precision"],
        "recall_after": baseline_result["after_recall"],
        "accuracy_after": baseline_result["after_accuracy"],
    },
    {
        "experiment": "MLM random -> NER",
        "f1_before": mlm_then_ner_result["before_f1"],
        "f1_after": mlm_then_ner_result["after_f1"],
        "precision_after": mlm_then_ner_result["after_precision"],
        "recall_after": mlm_then_ner_result["after_recall"],
        "accuracy_after": mlm_then_ner_result["after_accuracy"],
    },
]).sort_values("f1_after", ascending=False).reset_index(drop=True)

compare_df

,experiment,f1_before,f1_after,precision_after,recall_after,accuracy_after
0,baseline NER,0.018668,0.983239,0.981641,0.984843,0.996068
1,MLM random -> NER,0.018800,0.979232,0.976895,0.981581,0.995520


### Вывод

Дополнительное mlm-предобучение с обычным случайным маскированием не улучшило качество NER-модели.  
На validation baseline показал `f1 = 0.983`, тогда как схема MLM random -> NER дала `f1 = 0.979`.  
Вероятно, исходный энкодер уже был достаточно хорошо адаптирован к русскому языку, а baseline finetuning оказался почти оптимальным для данного корпуса.

## MLM с Whole Word Masking

In [23]:
# collator и MLM с whole word masking
wwm_collator = DataCollatorForWholeWordMask(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=0.15,
)

mlm_wwm_result = pretrain_mlm(
    model_name_or_path=BASE_MODEL,
    train_dataset=mlm_tokenized["train"],
    eval_dataset=mlm_tokenized["validation"],
    run_name="mlm_whole_word_masking",
    collator=wwm_collator,
    learning_rate=5e-5,
    num_train_epochs=MLM_EPOCHS,
    batch_size=MLM_BATCH_SIZE,
)

mlm_wwm_result

/home/mlcore/conda/lib/python3.12/site-packages/transformers/training_args.py:1568: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
/tmp/ipykernel_363/2600747489.py:35: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
We strongly recommend passing in an `attention_mask` since your input_ids may be padded. See https://huggingface.co/docs/transformers/troubleshooting#incorrect-output-when-padding-tokens-arent-masked.


Epoch,Training Loss,Validation Loss,Model Preparation Time
1,1.985600,1.958263,0.002700
2,1.746400,1.852723,0.002700


There were missing keys in the checkpoint model loaded: ['cls.predictions.decoder.weight', 'cls.predictions.decoder.bias'].


{'run_name': 'mlm_whole_word_masking',
 'mlm_loss_before': 2.966940402984619,
 'mlm_loss_after': 1.8279972076416016,
 'model_path': './hw3_ner_runs/mlm_whole_word_masking'}

## NER после whole word masking

In [24]:
wwm_then_ner_result = train_ner_model(
    model_name_or_path=mlm_wwm_result["model_path"], 
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["validation"], 
    run_name="mlm_wwm_then_ner",
    learning_rate=2e-5,
    num_train_epochs=NER_EPOCHS,
    batch_size=BATCH_SIZE,
)

wwm_then_ner_result

Some weights of BertForTokenClassification were not initialized from the model checkpoint at ./hw3_ner_runs/mlm_whole_word_masking and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/home/mlcore/conda/lib/python3.12/site-packages/transformers/training_args.py:1568: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
/tmp/ipykernel_363/3265979145.py:33: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Model Preparation Time,Precision,Recall,F1,Accuracy
1,0.033200,0.025293,0.002700,0.961231,0.970453,0.965820,0.992865
2,0.015900,0.018794,0.002700,0.969456,0.980430,0.974912,0.994724
3,0.007000,0.021404,0.002700,0.978899,0.979087,0.978993,0.995122
4,0.003100,0.024387,0.002700,0.977382,0.978319,0.977850,0.995089
5,0.003700,0.022049,0.002700,0.978736,0.980238,0.979486,0.995420


{'run_name': 'mlm_wwm_then_ner',
 'before_f1': 0.012848951575775333,
 'after_f1': 0.9794861963190183,
 'before_precision': 0.007210973833795274,
 'after_precision': 0.978735632183908,
 'before_recall': 0.0589025326170376,
 'after_recall': 0.9802379125095932,
 'before_accuracy': 0.075728413088206,
 'after_accuracy': 0.9954204552996615,
 'model_path': './hw3_ner_runs/mlm_wwm_then_ner'}

In [25]:
# сравниваем
compare_df = pd.DataFrame([
    {
        "experiment": "baseline NER",
        "f1_before": baseline_result["before_f1"],
        "f1_after": baseline_result["after_f1"],
        "precision_after": baseline_result["after_precision"],
        "recall_after": baseline_result["after_recall"],
        "accuracy_after": baseline_result["after_accuracy"],
    },
    {
        "experiment": "MLM random -> NER",
        "f1_before": mlm_then_ner_result["before_f1"],
        "f1_after": mlm_then_ner_result["after_f1"],
        "precision_after": mlm_then_ner_result["after_precision"],
        "recall_after": mlm_then_ner_result["after_recall"],
        "accuracy_after": mlm_then_ner_result["after_accuracy"],
    },
    {
        "experiment": "MLM whole word -> NER",
        "f1_before": wwm_then_ner_result["before_f1"],
        "f1_after": wwm_then_ner_result["after_f1"],
        "precision_after": wwm_then_ner_result["after_precision"],
        "recall_after": wwm_then_ner_result["after_recall"],
        "accuracy_after": wwm_then_ner_result["after_accuracy"],
    },
]).sort_values("f1_after", ascending=False).reset_index(drop=True)

compare_df

,experiment,f1_before,f1_after,precision_after,recall_after,accuracy_after
0,baseline NER,0.018668,0.983239,0.981641,0.984843,0.996068
1,MLM whole word -> NER,0.012849,0.979486,0.978736,0.980238,0.995420
2,MLM random -> NER,0.018800,0.979232,0.976895,0.981581,0.995520


После выполнения эксперимента с Whole Word Masking, мы видим, что результат не дал заметного улучшения в задаче NER.
Этот результат показывает, что маскирование целых слов в данном случае не улучшает модель, а даже несколько ухудшает результат. Это может происходить по причине того, что baseline уже сильный: модель была уже хорошо настроена на задачу, и дополнительные изменения не привели к улучшению.

## Concept Masking

In [26]:
# функция, которая находит сущности в одном примере и возвращает их
def add_entity_spans(example):
    spans = []
    for token, label in zip(example["tokens"], example["ner_tags_str"]):
        spans.append((token, label))
    return spans

# маскирует сущности в тексте с вероятностью prob
def concept_masking(examples, tokenizer, prob=0.15):
    all_masked_tokens = []
    for example in examples["tokens"]:
        mask_sentence = " ".join([token if random.random() > prob else "[MASK]" for token in example])
        all_masked_tokens.append(mask_sentence)
    
    return {"text_for_mlm": all_masked_tokens}

concept_masked_ds = ds.map(
    concept_masking,
    batched=True,
    fn_kwargs={"tokenizer": tokenizer},  # передаем tokenizer в функцию
)

def tokenize_concept_masked(examples):
    return tokenizer(
        examples["text_for_mlm"],
        truncation=True,
        padding=True,
        max_length=MAX_LENGTH,
        return_tensors="pt",
    )

concept_masked_tokenized = tokenizer(
    concept_masked_ds["train"]["text_for_mlm"],
    truncation=True,
    padding=True,
    max_length=MAX_LENGTH,
    return_tensors="pt",
)

concept_masked_tokenized = Dataset.from_dict(concept_masked_tokenized)

concept_masked_tokenized

Map:   0%|          | 0/7746 [00:00<?, ? examples/s]

Map:   0%|          | 0/2582 [00:00<?, ? examples/s]

Map:   0%|          | 0/2582 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 7746
})

In [27]:
concept_masked_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=0.15,
)

concept_masked_result = pretrain_mlm(
    model_name_or_path=BASE_MODEL,               
    train_dataset=concept_masked_tokenized,    
    eval_dataset=concept_masked_tokenized,      
    run_name="concept_masking",              
    collator=concept_masked_collator,           
    learning_rate=5e-5,
    num_train_epochs=MLM_EPOCHS,
    batch_size=MLM_BATCH_SIZE,
)

concept_masked_result

/home/mlcore/conda/lib/python3.12/site-packages/transformers/training_args.py:1568: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
/tmp/ipykernel_363/2600747489.py:35: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Model Preparation Time
1,2.106400,1.800288,0.002900
2,1.967600,1.711103,0.002900


There were missing keys in the checkpoint model loaded: ['cls.predictions.decoder.weight', 'cls.predictions.decoder.bias'].


{'run_name': 'concept_masking',
 'mlm_loss_before': 2.4487760066986084,
 'mlm_loss_after': 1.663066029548645,
 'model_path': './hw3_ner_runs/concept_masking'}

## NER после concept masking

In [28]:
concept_masked_ner_result = train_ner_model(
    model_name_or_path=concept_masked_result["model_path"],  
    train_dataset=tokenized_ds["train"],                   
    eval_dataset=tokenized_ds["validation"],          
    run_name="concept_masking_then_ner",                    
    learning_rate=2e-5,
    num_train_epochs=NER_EPOCHS,
    batch_size=BATCH_SIZE,
)

concept_masked_ner_result

Some weights of BertForTokenClassification were not initialized from the model checkpoint at ./hw3_ner_runs/concept_masking and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/home/mlcore/conda/lib/python3.12/site-packages/transformers/training_args.py:1568: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
/tmp/ipykernel_363/3265979145.py:33: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Model Preparation Time,Precision,Recall,F1,Accuracy
1,0.033800,0.023247,0.002800,0.957415,0.974866,0.966061,0.993097
2,0.014500,0.018214,0.002800,0.972860,0.983500,0.978151,0.995337
3,0.006700,0.021008,0.002800,0.975443,0.983116,0.979264,0.995371
4,0.004300,0.019677,0.002800,0.974296,0.981773,0.978020,0.995852
5,0.003200,0.021136,0.002800,0.976900,0.981773,0.979330,0.995868


{'run_name': 'concept_masking_then_ner',
 'before_f1': 0.022317801672640383,
 'after_f1': 0.9793301435406698,
 'before_precision': 0.012746328948086687,
 'after_precision': 0.9768995799923635,
 'before_recall': 0.08960092095165004,
 'after_recall': 0.9817728319263239,
 'before_accuracy': 0.2297404924669808,
 'after_accuracy': 0.9958684542377381,
 'model_path': './hw3_ner_runs/concept_masking_then_ner'}

In [29]:
compare_df = pd.DataFrame([
    {
        "experiment": "baseline NER",
        "f1_before": baseline_result["before_f1"],
        "f1_after": baseline_result["after_f1"],
        "precision_after": baseline_result["after_precision"],
        "recall_after": baseline_result["after_recall"],
        "accuracy_after": baseline_result["after_accuracy"],
    },
    {
        "experiment": "MLM random -> NER",
        "f1_before": mlm_then_ner_result["before_f1"],
        "f1_after": mlm_then_ner_result["after_f1"],
        "precision_after": mlm_then_ner_result["after_precision"],
        "recall_after": mlm_then_ner_result["after_recall"],
        "accuracy_after": mlm_then_ner_result["after_accuracy"],
    },
    {
        "experiment": "MLM whole word -> NER",
        "f1_before": wwm_then_ner_result["before_f1"],
        "f1_after": wwm_then_ner_result["after_f1"],
        "precision_after": wwm_then_ner_result["after_precision"],
        "recall_after": wwm_then_ner_result["after_recall"],
        "accuracy_after": wwm_then_ner_result["after_accuracy"],
    },
    {
        "experiment": "concept masking -> NER",
        "f1_before": concept_masked_ner_result["before_f1"],
        "f1_after": concept_masked_ner_result["after_f1"],
        "precision_after": concept_masked_ner_result["after_precision"],
        "recall_after": concept_masked_ner_result["after_recall"],
        "accuracy_after": concept_masked_ner_result["after_accuracy"],
    },
]).sort_values("f1_after", ascending=False).reset_index(drop=True)

compare_df

,experiment,f1_before,f1_after,precision_after,recall_after,accuracy_after
0,baseline NER,0.018668,0.983239,0.981641,0.984843,0.996068
1,MLM whole word -> NER,0.012849,0.979486,0.978736,0.980238,0.995420
2,concept masking -> NER,0.022318,0.979330,0.976900,0.981773,0.995868
3,MLM random -> NER,0.018800,0.979232,0.976895,0.981581,0.995520


На основе этих экспериментов можно заключить, что baseline NER уже даёт очень хорошее качество, и дополнительные шаги не улучшают модель

## Выводы

- **Baseline NER** показал отличные результаты с `F1 = 0.983`, что делает дальнейшие улучшения сложными.
- **MLM random masking** и **MLM whole word masking** не дали значительного улучшения и показали небольшое ухудшение.
- **Concept masking** также не улучшил качество модели и немного снизил **F1** по сравнению с baseline.
- Это указывает на то, что модель была уже достаточно хорошо адаптирована к задаче NER, и дополнительные изменения не всегда приводят к улучшению.

## Синтетическая разметка для дообучения модели

In [30]:
from transformers import pipeline

model_name = "Gherman/bert-base-NER-Russian"
ner_pipeline = pipeline("ner", model=model_name, tokenizer=model_name)


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/709M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.


In [32]:
# проверим работу модели
test_text = "Юрий Гагарин - советский космонавт"
predictions = ner_pipeline(test_text)

predictions

[{'entity': 'U-FIRST_NAME',
  'score': 0.99482876,
  'index': 1,
  'word': 'Юрий',
  'start': 0,
  'end': 4},
 {'entity': 'U-LAST_NAME',
  'score': 0.99755424,
  'index': 2,
  'word': 'Г',
  'start': 5,
  'end': 6},
 {'entity': 'U-LAST_NAME',
  'score': 0.9974432,
  'index': 3,
  'word': '##ага',
  'start': 6,
  'end': 9},
 {'entity': 'U-LAST_NAME',
  'score': 0.99686414,
  'index': 4,
  'word': '##рин',
  'start': 9,
  'end': 12}]

In [49]:
from corus import load_lenta

N_SAMPLES = 10000

DATA_PATH = Path("lenta-ru-news.csv.gz")
print(DATA_PATH)

if not DATA_PATH.exists():
    !wget -O lenta-ru-news.csv.gz https://github.com/yutkin/Lenta.Ru-News-Dataset/releases/download/v1.0/lenta-ru-news.csv.gz

assert DATA_PATH.exists(), "Не удалось скачать lenta-ru-news.csv.gz"
print(DATA_PATH)

rows = []

for rec in tqdm(load_lenta(str(DATA_PATH)), desc="Loading lenta"):
    rows.append({
        "title": rec.title,
        "text": rec.text,
        "topic": rec.topic
    })

df = pd.DataFrame(rows)

df["topic"] = df["topic"].astype(str).str.strip()
df = df[df["topic"] != ""].copy()
df["title"] = df["title"].fillna("").astype(str).str.strip()
df["text"] = df["text"].fillna("").astype(str).str.strip()

print(df.shape)
print(df["topic"].nunique())

if len(df) > N_SAMPLES:
    df_sample = df.sample(n=N_SAMPLES, random_state=42).copy()
else:
    df_sample = df.copy()

df_sample = df_sample.reset_index(drop=True)

print(df_sample.shape)

lenta-ru-news.csv.gz
lenta-ru-news.csv.gz


Loading lenta: 0it [00:00, ?it/s]

(739148, 3)
23
(10000, 3)


## Подготовка псевдоразметки

Для синтетической разметки объединяем заголовок и текст новости, после чего прогоняем получившиеся тексты через NER-модель.
Предсказания teacher-модели затем приводятся к схеме меток PER / ORG / LOC, совместимой с основным датасетом.

In [50]:
df_sample["full_text"] = (
    df_sample["title"].fillna("").astype(str).str.strip() + ". " +
    df_sample["text"].fillna("").astype(str).str.strip()
).str.strip()

df_sample["full_text"] = df_sample["full_text"].str.replace(r"\s+", " ", regex=True)

print(df_sample[["title", "full_text"]].head(2))
print("num rows:", len(df_sample))

                                               title  \
0  В Кабуле при нападении талибов убиты восемь со...   
1  Из-за скандала с утечкой информации Буш остане...   

                                           full_text  
0  В Кабуле при нападении талибов убиты восемь со...  
1  Из-за скандала с утечкой информации Буш остане...  
num rows: 10000


In [52]:
#токенизация текстов для teacher-модели
def whitespace_tokenize(text, max_words=256):
    return text.split()[:max_words]

df_sample["tokens"] = df_sample["full_text"].apply(lambda x: whitespace_tokenize(x, max_words=256))
df_sample = df_sample[df_sample["tokens"].map(len) > 0].reset_index(drop=True)

print(df_sample[["tokens"]].head(2))
print("num rows after tokenization:", len(df_sample))

                                              tokens
0  [В, Кабуле, при, нападении, талибов, убиты, во...
1  [Из-за, скандала, с, утечкой, информации, Буш,...
num rows after tokenization: 10000


In [53]:
from collections import Counter

teacher_label_counter = Counter()

for i in range(200):
    text = " ".join(df_sample.loc[i, "tokens"])
    preds = ner_pipeline(text)
    teacher_label_counter.update([p["entity"] for p in preds])

teacher_label_counter.most_common(50)

[('U-LAST_NAME', 2934),
 ('U-CITY', 1105),
 ('U-FIRST_NAME', 1080),
 ('U-COUNTRY', 1030),
 ('U-REGION', 274),
 ('U-DISTRICT', 102),
 ('L-CITY', 66),
 ('B-COUNTRY', 46),
 ('L-COUNTRY', 42),
 ('U-MIDDLE_NAME', 33),
 ('I-CITY', 31),
 ('B-STREET', 28),
 ('B-LAST_NAME', 22),
 ('L-STREET', 21),
 ('L-LAST_NAME', 20),
 ('B-CITY', 17),
 ('U-STREET', 11),
 ('I-COUNTRY', 9),
 ('B-REGION', 7),
 ('I-LAST_NAME', 4),
 ('B-DISTRICT', 3),
 ('L-DISTRICT', 3),
 ('I-HOUSE', 3),
 ('I-STREET', 2),
 ('U-HOUSE', 1),
 ('L-HOUSE', 1)]

In [55]:
def normalize_teacher_tag(raw_tag):
    tag = raw_tag.upper()

    if any(x in tag for x in [
        "FIRST_NAME", "LAST_NAME", "MIDDLE_NAME", "PERSON"]):
        return "PER"

    if any(x in tag for x in [
        "CITY", "COUNTRY", "REGION", "DISTRICT", "STREET", "HOUSE", "LOCATION"]):
        return "LOC"

    if any(x in tag for x in [
        "ORG", "ORGANIZATION", "COMPANY"]):
        return "ORG"

    return None


def teacher_preds_to_bio(tokens, preds):
    labels = ["O"] * len(tokens)

    spans = []
    pos = 0
    for tok in tokens:
        spans.append((pos, pos + len(tok)))
        pos += len(tok) + 1

    used = set()

    for pred in preds:
        ent_type = normalize_teacher_tag(pred["entity"])
        if ent_type is None:
            continue

        start_char = pred["start"]
        end_char = pred["end"]

        covered = []
        for i, (s, e) in enumerate(spans):
            if not (e <= start_char or s >= end_char):
                covered.append(i)

        if not covered:
            continue

        covered = [i for i in covered if i not in used]
        if not covered:
            continue

        labels[covered[0]] = f"B-{ent_type}"
        for i in covered[1:]:
            labels[i] = f"I-{ent_type}"

        used.update(covered)

    return labels

In [57]:
test_tokens = df_sample.loc[0, "tokens"]
test_text = " ".join(test_tokens)
test_preds = ner_pipeline(test_text)
test_bio = teacher_preds_to_bio(test_tokens, test_preds)

print(test_text[:500])
print(test_preds[:10])
print(list(zip(test_tokens[:40], test_bio[:40])))

В Кабуле при нападении талибов убиты восемь сотрудников НАТО. Восемь членов миссии НАТО в Афганистане убиты в результате нападения талибов на базу альянса около аэропорта Кабула. Об этом сообщает Reuters. Имена и национальности погибших не уточняются. Агентство отмечает, что нападение на базу НАТО произошло на следующий день после серии терактов в Афганистане. Вечером 7 августа террорист-смертник устроил взрыв на территории полицейской академии в Кабуле. По последним данным, которые приводит Reu
[{'entity': 'U-CITY', 'score': 0.99708873, 'index': 2, 'word': 'К', 'start': 2, 'end': 3}, {'entity': 'U-CITY', 'score': 0.9972383, 'index': 3, 'word': '##абул', 'start': 3, 'end': 7}, {'entity': 'U-COUNTRY', 'score': 0.9940388, 'index': 25, 'word': 'А', 'start': 90, 'end': 91}, {'entity': 'U-COUNTRY', 'score': 0.995076, 'index': 26, 'word': '##ф', 'start': 91, 'end': 92}, {'entity': 'U-COUNTRY', 'score': 0.9941215, 'index': 27, 'word': '##ган', 'start': 92, 'end': 95}, {'entity': 'U-COUNTRY', 

In [58]:
pseudo_rows = []

for tokens in tqdm(df_sample["tokens"], total=len(df_sample), desc="Pseudo-labeling"):
    text = " ".join(tokens)
    preds = ner_pipeline(text)
    bio_tags = teacher_preds_to_bio(tokens, preds)

    if len(tokens) != len(bio_tags):
        continue

    pseudo_rows.append({
        "tokens": tokens,
        "ner_tags_str": bio_tags,
        "ner_tags": [label2id[tag] for tag in bio_tags],
    })

print("pseudo rows:", len(pseudo_rows))
print(pseudo_rows[0])

Pseudo-labeling:   0%|          | 0/10000 [00:00<?, ?it/s]

pseudo rows: 10000
{'tokens': ['В', 'Кабуле', 'при', 'нападении', 'талибов', 'убиты', 'восемь', 'сотрудников', 'НАТО.', 'Восемь', 'членов', 'миссии', 'НАТО', 'в', 'Афганистане', 'убиты', 'в', 'результате', 'нападения', 'талибов', 'на', 'базу', 'альянса', 'около', 'аэропорта', 'Кабула.', 'Об', 'этом', 'сообщает', 'Reuters.', 'Имена', 'и', 'национальности', 'погибших', 'не', 'уточняются.', 'Агентство', 'отмечает,', 'что', 'нападение', 'на', 'базу', 'НАТО', 'произошло', 'на', 'следующий', 'день', 'после', 'серии', 'терактов', 'в', 'Афганистане.', 'Вечером', '7', 'августа', 'террорист-смертник', 'устроил', 'взрыв', 'на', 'территории', 'полицейской', 'академии', 'в', 'Кабуле.', 'По', 'последним', 'данным,', 'которые', 'приводит', 'Reuters,', 'погибли', 'более', '40', 'человек.', 'Ответственность', 'за', 'теракт', 'взяло', 'на', 'себя', 'движение', '«Талибан».', 'Утром', 'пятницы', 'возле', 'военной', 'части', 'в', 'столице', 'Афганистана', 'взорвался', 'начиненный', 'взрывчаткой', 'автомоби

In [59]:
# синтетический датасет

pseudo_dataset = Dataset.from_list(pseudo_rows)
pseudo_dataset

Dataset({
    features: ['tokens', 'ner_tags_str', 'ner_tags'],
    num_rows: 10000
})

In [60]:
def has_any_entity(example):
    return any(tag != "O" for tag in example["ner_tags_str"])

num_with_entities = sum(has_any_entity(x) for x in pseudo_rows)
print("rows with at least one entity:", num_with_entities)
print("share:", round(num_with_entities / len(pseudo_rows), 4))

rows with at least one entity: 9754
share: 0.9754


In [61]:
# токенизация синтетического датасета
pseudo_tokenized = pseudo_dataset.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=pseudo_dataset.column_names,
)

pseudo_tokenized

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
    num_rows: 10000
})

In [62]:
# обьединяем голд и синтетику
mixed_train = concatenate_datasets([
    tokenized_ds["train"],
    pseudo_tokenized,
]).shuffle(seed=42)

mixed_train

Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
    num_rows: 17746
})

In [63]:
# обучение на микс датасете
pseudo_label_result = train_ner_model(
    model_name_or_path=BASE_MODEL,
    train_dataset=mixed_train,
    eval_dataset=tokenized_ds["validation"],
    run_name="gold_plus_pseudolabels",
    learning_rate=2e-5,
    num_train_epochs=NER_EPOCHS,
    batch_size=BATCH_SIZE,
)

pseudo_label_result

Some weights of BertForTokenClassification were not initialized from the model checkpoint at DeepPavlov/rubert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/home/mlcore/conda/lib/python3.12/site-packages/transformers/training_args.py:1568: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
/tmp/ipykernel_363/3265979145.py:33: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Model Preparation Time,Precision,Recall,F1,Accuracy
1,0.022800,0.030842,0.003100,0.932916,0.971220,0.951683,0.989497
2,0.016700,0.021186,0.003100,0.962760,0.977168,0.969910,0.994043
3,0.011700,0.020402,0.003100,0.966484,0.979279,0.972839,0.994574
4,0.007000,0.019574,0.003100,0.969207,0.978319,0.973742,0.994956
5,0.005100,0.020066,0.003100,0.970711,0.979279,0.974976,0.995055


{'run_name': 'gold_plus_pseudolabels',
 'before_f1': 0.013852997246346114,
 'after_f1': 0.9749761222540592,
 'before_precision': 0.007786085051669127,
 'after_precision': 0.9707112970711297,
 'before_recall': 0.06273983115886415,
 'after_recall': 0.9792785878741366,
 'before_accuracy': 0.12270193137319971,
 'after_accuracy': 0.9950554191278954,
 'model_path': './hw3_ner_runs/gold_plus_pseudolabels'}

In [64]:
final_compare_df = pd.DataFrame([
    {
        "experiment": "baseline NER",
        "f1_before": baseline_result["before_f1"],
        "f1_after": baseline_result["after_f1"],
        "precision_after": baseline_result["after_precision"],
        "recall_after": baseline_result["after_recall"],
        "accuracy_after": baseline_result["after_accuracy"],
    },
    {
        "experiment": "MLM random -> NER",
        "f1_before": mlm_then_ner_result["before_f1"],
        "f1_after": mlm_then_ner_result["after_f1"],
        "precision_after": mlm_then_ner_result["after_precision"],
        "recall_after": mlm_then_ner_result["after_recall"],
        "accuracy_after": mlm_then_ner_result["after_accuracy"],
    },
    {
        "experiment": "MLM whole word -> NER",
        "f1_before": wwm_then_ner_result["before_f1"],
        "f1_after": wwm_then_ner_result["after_f1"],
        "precision_after": wwm_then_ner_result["after_precision"],
        "recall_after": wwm_then_ner_result["after_recall"],
        "accuracy_after": wwm_then_ner_result["after_accuracy"],
    },
    {
        "experiment": "concept masking -> NER",
        "f1_before": concept_masked_ner_result["before_f1"],
        "f1_after": concept_masked_ner_result["after_f1"],
        "precision_after": concept_masked_ner_result["after_precision"],
        "recall_after": concept_masked_ner_result["after_recall"],
        "accuracy_after": concept_masked_ner_result["after_accuracy"],
    },
    {
        "experiment": "gold + pseudo labels",
        "f1_before": pseudo_label_result["before_f1"],
        "f1_after": pseudo_label_result["after_f1"],
        "precision_after": pseudo_label_result["after_precision"],
        "recall_after": pseudo_label_result["after_recall"],
        "accuracy_after": pseudo_label_result["after_accuracy"],
    },
]).sort_values("f1_after", ascending=False).reset_index(drop=True)

final_compare_df

,experiment,f1_before,f1_after,precision_after,recall_after,accuracy_after
0,baseline NER,0.018668,0.983239,0.981641,0.984843,0.996068
1,MLM whole word -> NER,0.012849,0.979486,0.978736,0.980238,0.995420
2,concept masking -> NER,0.022318,0.979330,0.976900,0.981773,0.995868
3,MLM random -> NER,0.018800,0.979232,0.976895,0.981581,0.995520
4,gold + pseudo labels,0.013853,0.974976,0.970711,0.979279,0.995055


## Итоговые выводы

В работе была решена задача NER для корпуса FactRuEval-2016.  
В качестве базовой модели был выбран DeepPavlov/rubert-base-cased, поскольку это предобученный русскоязычный BERT-энкодер, хорошо подходящий для sequence labeling-задач на русском языке.  
Датасет был подготовлен к обучению: выполнена токенизация, выравнивание BIO-меток после сабтокенизации и настройка вычисления NER-метрик (`precision`, `recall`, `F1`, `accuracy`).

### 1. Baseline

Базовое дообучение модели на train-части FactRuEval-2016 показало лучший результат среди всех экспериментов:

- **baseline NER**: `F1 = 0.983239`

Это означает, что исходный энкодер уже очень хорошо подходит для данной задачи, а обычный fine-tuning оказался крайне сильным решением.

### 2. MLM-предобучение перед NER

Были проверены два варианта дополнительного MLM-предобучения на train-части корпуса:

- **MLM random -> NER**: `F1 = 0.979232`
- **MLM whole word -> NER**: `F1 = 0.979486`

Оба варианта показали качество немного ниже baseline.  
Whole Word Masking оказался немного лучше обычного random masking, но всё равно не превзошёл прямое дообучение на NER.

Это можно объяснить тем, что:
- исходный `RuBERT` уже хорошо адаптирован к русскому языку;
- сам baseline fine-tuning уже очень эффективно подстраивает модель под задачу;
- дополнительный MLM-этап в данном случае не дал полезного прироста, а скорее немного сместил представления модели в сторону языкового моделирования.

### 3. Concept masking

Дополнительно был проверен вариант **concept masking**, где при MLM-предобучении усиливался фокус на сущностях:

- **concept masking -> NER**: `F1 = 0.979330`

Этот результат также уступил baseline.  
Следовательно, даже более целенаправленное маскирование сущностей не улучшило итоговое качество на FactRuEval-2016.

### 4. Синтетическая разметка

Для генерации синтетической разметки был выбран внешний новостной корпус Lenta.ru.  
Из него была взята случайная подвыборка из **10000** текстов.  
В качестве teacher-модели использовалась внешняя русскоязычная NER-модель **`Gherman/bert-base-NER-Russian`**, предсказания которой были приведены к целевой схеме меток `PER / ORG / LOC` и использованы как псевдоразметка.

После объединения gold-данных и synthetic data был получен результат:

- **gold + pseudo labels**: `F1 = 0.974976`

Этот подход показал худший результат среди всех улучшений и тоже не превзошёл baseline.

С высокой вероятностью это связано с качеством самой синтетической разметки:
- teacher-модель хорошо размечала `PER` и `LOC`,
- но почти не давала качественных `ORG`,
- кроме того, её собственная схема сущностей отличалась от схемы FactRuEval-2016 и требовала грубого приведения меток,
- из-за этого в synthetic data появился дополнительный шум, который ухудшил обучение.

### 5. Финальное сравнение подходов

Итоговый рейтинг экспериментов по `F1`:

1. **baseline NER** — `0.983239`
2. **MLM whole word -> NER** — `0.979486`
3. **concept masking -> NER** — `0.979330`
4. **MLM random -> NER** — `0.979232`
5. **gold + pseudo labels** — `0.974976`

### Общий вывод

Лучшим подходом в данной работе оказалось **обычное дообучение `DeepPavlov/rubert-base-cased` на задаче NER без дополнительных усложнений**.  
Ни один из проверенных способов улучшения — ни MLM-предобучение, ни Whole Word Masking, ни Concept Masking, ни синтетическая разметка — не смог превзойти baseline.

Это важный практический результат: если базовая модель и так хорошо соответствует языку и задаче, то дополнительные этапы обучения не гарантируют улучшение качества и иногда даже приводят к деградации.

### Что можно улучшить в дальнейших экспериментах

В дальнейшем можно попробовать следующие направления:

- использовать более сильную teacher-модель для синтетической разметки;
- фильтровать synthetic data по confidence-score, чтобы уменьшить шум;
- добавлять synthetic data не целиком, а в ограниченной пропорции к gold-данным;
- попробовать более сильные русскоязычные encoder-модели;
- провести отдельный подбор гиперпараметров для MLM-этапа;
- попробовать early stopping и более мягкое количество эпох;
- провести анализ ошибок по типам сущностей (`PER`, `ORG`, `LOC`) отдельно.

### Главный итог работы: 

**Для данного корпуса наиболее эффективным оказался сильный baseline без дополнительных этапов предобучения и без псевдоразметки**.